# 1.0 Setup: connecting Google Drive, loading pandas and defining the project folders

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import os

BASE  = '/content/drive/MyDrive/cs2-regional-demand'
RAW   = f'{BASE}/data/raw'
CLEAN = f'{BASE}/data/clean'
FINAL = f'{BASE}/data/final'

# CHECK: list every file in data/raw with its size
for f in sorted(os.listdir(RAW)):
    size_kb = os.path.getsize(f'{RAW}/{f}') / 1024
    print(f'{f:45s} {size_kb:8.1f} KB')

Mounted at /content/drive
d1_population_2010_2022.csv                       24.6 KB
d2_disability_2016.xlsx                           18.2 KB
d3_moh_rehab_hospitals.xlsx                       23.7 KB
d4_gastat_disability_2023.xlsx                    19.7 KB
d5_gastat_healthcare_2024.xlsx                     9.0 KB


# 1.1a Reviewing how each source file spells the 13 region names

In [2]:
names = {
    'D1':  pd.read_csv(f'{RAW}/d1_population_2010_2022.csv')['Region'],
    'D2':  pd.read_excel(f'{RAW}/d2_disability_2016.xlsx', sheet_name='t2_prevalence_2016')['region'],
    'D2b': pd.read_excel(f'{RAW}/d2_disability_2016.xlsx', sheet_name='t3_type_by_severity_2016')['region'],
    'D3':  pd.read_excel(f'{RAW}/d3_moh_rehab_hospitals.xlsx', sheet_name='hospitals')['admin_region'],
    'D4':  pd.read_excel(f'{RAW}/d4_gastat_disability_2023.xlsx', sheet_name='regional_household_share')['region_in_source'],
    'D5':  pd.read_excel(f'{RAW}/d5_gastat_healthcare_2024.xlsx', sheet_name='regional')['region_in_source'],
}

for file, col in names.items():
    unique_names = sorted(col.dropna().unique())
    print(f'{file}: {len(unique_names)} names')
    print('   ', unique_names)

D1: 13 names
    ['Al Bahah', 'Al Jawf', 'Al Madinah Al Munawwarah', 'Al Qaseem', 'Ar Riyadh', 'Aseer', 'Eastern Region', 'Hail', 'Jazan', 'Makkah Al Mukarramah', 'Najran', 'Northern Borders', 'Tabuk']
D2: 13 names
    ['Al-Bahah', 'Al-Jouf', 'Al-Qassim', 'Asir', 'Eastern Region', 'Hail', 'Jizan', 'Madinah', 'Makkah', 'Najran', 'Northern Border', 'Riyadh', 'Tabuk']
D2b: 13 names
    ['Al-Bahah', 'Al-Jouf', 'Al-Qassim', 'Asir', 'Eastern Region', 'Hail', 'Jizan', 'Madinah', 'Makkah', 'Najran', 'Northern Border', 'Riyadh', 'Tabuk']
D3: 13 names
    ['Al Bahah', 'Al Jawf', 'Al Qaseem', 'Asir', 'Eastern Region', 'Hail', 'Jazan', 'Madinah', 'Makkah', 'Najran', 'Northern Borders', 'Riyadh', 'Tabuk']
D4: 14 names
    ['Al-Baha', 'Al-Jouf', 'Al-Madinah Al-Monawarah', 'Al-Qaseem', 'Al-Riyadh', 'Aseer', 'Eastern Region', 'Hail', 'Jazan', 'Makkah Al-Mukarramah', 'Najran', 'Northern Borders', 'TOTAL', 'Tabuk']
D5: 13 names
    ['Al-Baha', 'Al-Jouf', 'Aseer', 'Eastern Region', 'Hail', 'Jazan', 'Madi

# 1.1b Building a region lookup that maps every spelling to one official GASTAT name

In [3]:
lookup = {
    'Al Bahah':                 ['Al-Bahah', 'Al-Baha'],
    'Al Jawf':                  ['Al-Jouf'],
    'Al Madinah Al Munawwarah': ['Madinah', 'Al-Madinah Al-Monawarah'],
    'Al Qaseem':                ['Al-Qassim', 'Qassim', 'Al-Qaseem'],
    'Ar Riyadh':                ['Riyadh', 'Al-Riyadh'],
    'Aseer':                    ['Asir'],
    'Eastern Region':           [],
    'Hail':                     [],
    'Jazan':                    ['Jizan'],
    'Makkah Al Mukarramah':     ['Makkah', 'Makkah Al-Mukarramah'],
    'Najran':                   [],
    'Northern Borders':         ['Northern Border'],
    'Tabuk':                    [],
}

# turn it into a 2-column table: variant -> official region
rows = []
for official, variants in lookup.items():
    for v in [official] + variants:
        rows.append({'variant': v, 'region': official})
region_lookup = pd.DataFrame(rows)

# CHECK 1: exactly the 13 official D1 names, and no spelling listed twice
official_names = set(names['D1'].unique())
assert set(lookup) == official_names, 'Official names do not match D1'
assert region_lookup['variant'].is_unique, 'A spelling is listed twice'

# CHECK 2: every spelling in every file finds a match
all_ok = True
for file, col in names.items():
    found = set(col.dropna().unique()) - {'TOTAL'}
    missing = sorted(found - set(region_lookup['variant']))
    print(f'{file}: unmatched = {missing if missing else "none"}')
    if missing:
        all_ok = False
assert all_ok, 'Some names are not in the lookup - add them above'

# SAVE
region_lookup.to_csv(f'{CLEAN}/region_lookup.csv', index=False)
print(f'Saved region_lookup.csv: {len(region_lookup)} rows, {region_lookup.region.nunique()} regions')

D1: unmatched = none
D2: unmatched = none
D2b: unmatched = none
D3: unmatched = none
D4: unmatched = none
D5: unmatched = none
Saved region_lookup.csv: 28 rows, 13 regions


#1.2 Population: Saudi and non-Saudi residents by region, 2010–2022

In [4]:
# The region lookup from 1.1 is loaded once and reused in every following step
region_lookup = pd.read_csv(f'{CLEAN}/region_lookup.csv')
region_map = dict(zip(region_lookup['variant'], region_lookup['region']))

d1 = pd.read_csv(f'{RAW}/d1_population_2010_2022.csv')
print('Raw D1:', d1.shape, '| columns:', list(d1.columns))
print('Nationality values:', d1['Nationality'].unique())

# Region names are converted to the official spelling
d1['region'] = d1['Region'].map(region_map)

# Male and female are added together; Saudi and non-Saudi become two columns
pop = (d1.pivot_table(index=['region', 'Year'], columns='Nationality',
                      values='Population', aggfunc='sum')
         .reset_index()
         .rename(columns={'Year': 'year', 'Saudi': 'saudi', 'Non Saudi': 'non_saudi'}))
pop.columns.name = None
pop['total'] = pop['saudi'] + pop['non_saudi']
pop = pop[['region', 'year', 'saudi', 'non_saudi', 'total']].sort_values(['region', 'year'])

# Quality checks against the published 2022 census totals
y22 = pop[pop['year'] == 2022]
print('Rows:', len(pop), '| regions:', pop['region'].nunique(), '| years:', pop['year'].min(), '-', pop['year'].max())
print(f"2022 Saudi total: {y22['saudi'].sum():,}")
print(f"2022 total population: {y22['total'].sum():,}")
assert len(pop) == 169, 'Expected 13 regions x 13 years'
assert pop['region'].isna().sum() == 0, 'A region name did not match the lookup'
assert y22['saudi'].sum() == 18_792_262, 'Saudi 2022 total does not match the census'
assert y22['total'].sum() == 32_175_224, 'Total 2022 population does not match the census'
assert (pop[['saudi', 'non_saudi']] > 0).all().all(), 'Found a zero or negative count'

pop.to_csv(f'{CLEAN}/pop_clean.csv', index=False)
print('Saved pop_clean.csv')
print(y22.to_string(index=False))

Raw D1: (676, 5) | columns: ['Gender', 'Nationality', 'Region', 'Year', 'Population']
Nationality values: ['Saudi' 'Non Saudi']
Rows: 169 | regions: 13 | years: 2010 - 2022
2022 Saudi total: 18,792,262
2022 total population: 32,175,224
Saved pop_clean.csv
                  region  year   saudi  non_saudi   total
                Al Bahah  2022  251288      87886  339174
                 Al Jawf  2022  440264     155558  595822
Al Madinah Al Munawwarah  2022 1352146     785837 2137983
               Al Qaseem  2022  926490     409689 1336179
               Ar Riyadh  2022 4439210    4152538 8591748
                   Aseer  2022 1444688     579597 2024285
          Eastern Region  2022 2949854    2175400 5125254
                    Hail  2022  527885     218521  746406
                   Jazan  2022 1002779     402218 1404997
    Makkah Al Mukarramah  2022 4153723    3867740 8021463
                  Najran  2022  394976     197324  592300
        Northern Borders  2022  271358     10221

#1.3 Disability prevalence by region, 2016 (Bindawas & Vennu, Table 2)

In [5]:
# 1.3 Recomputing the 2016 regional disability rates from raw counts and checking them against the published paper

pop = pd.read_csv(f'{CLEAN}/pop_clean.csv')
t2 = pd.read_excel(f'{RAW}/d2_disability_2016.xlsx', sheet_name='t2_prevalence_2016')
print('Raw Table 2:', t2.shape, '| columns:', list(t2.columns))

prev = pd.DataFrame({
    'region':                 t2['region'].map(region_map),
    'saudi_pop_2016':         t2['saudi_population_2016'],
    'pwd_2016':               t2['persons_with_disability_2016'],
    'rate_per_100k_reported': t2['prevalence_per_100k'],
})

# Rates are recalculated from the counts rather than copied from the paper
prev['rate_per_100k'] = (prev['pwd_2016'] / prev['saudi_pop_2016'] * 100_000).round(1)
prev['rate_pct']      = (prev['pwd_2016'] / prev['saudi_pop_2016'] * 100).round(3)

# Quality checks against the totals published in the paper
nat_rate = prev['pwd_2016'].sum() / prev['saudi_pop_2016'].sum() * 100
gap = (prev['rate_per_100k'] - prev['rate_per_100k_reported']).abs().max()
print(f"Saudi population 2016 (survey): {prev['saudi_pop_2016'].sum():,}")
print(f"Persons with disability 2016:   {prev['pwd_2016'].sum():,}")
print(f'National rate 2016: {nat_rate:.2f}%')
print(f'Largest gap between recomputed and reported rate: {gap:.1f} per 100k')
assert prev['region'].isna().sum() == 0 and prev['region'].nunique() == 13, 'Region mismatch'
assert prev['saudi_pop_2016'].sum() == 20_064_970, 'Population total differs from the paper'
assert prev['pwd_2016'].sum() == 667_280, 'Disability total differs from the paper'
assert gap <= 1, 'Recomputed rates do not match the paper'
assert round(nat_rate, 2) == 3.33, 'National rate differs from the paper (3.33%)'

# Cross-check: the 2016 survey population compared with GASTAT's current 2016 estimate (D1)
d1_2016 = pop.loc[pop['year'] == 2016, ['region', 'saudi']].rename(columns={'saudi': 'saudi_pop_2016_D1'})
cross = prev[['region', 'saudi_pop_2016']].merge(d1_2016, on='region')
cross['survey_vs_D1_pct'] = ((cross['saudi_pop_2016'] / cross['saudi_pop_2016_D1'] - 1) * 100).round(1)
print(f"\nSaudi population 2016 - survey: {cross['saudi_pop_2016'].sum():,} | D1: {cross['saudi_pop_2016_D1'].sum():,}")
print(cross.to_string(index=False))

prev = prev.drop(columns='rate_per_100k_reported').sort_values('rate_per_100k', ascending=False)
prev.to_csv(f'{CLEAN}/prev2016_clean.csv', index=False)
print('\nSaved prev2016_clean.csv')
print(prev.to_string(index=False))

Raw Table 2: (13, 5) | columns: ['region', 'saudi_population_2016', 'persons_with_disability_2016', 'prevalence_per_100k', 'check_rate']
Saudi population 2016 (survey): 20,064,970
Persons with disability 2016:   667,280
National rate 2016: 3.33%
Largest gap between recomputed and reported rate: 0.5 per 100k

Saudi population 2016 - survey: 20,064,970 | D1: 16,355,312
                  region  saudi_pop_2016  saudi_pop_2016_D1  survey_vs_D1_pct
                   Tabuk          710699             559538              27.0
                 Al Jawf          373662             389853              -4.2
    Makkah Al Mukarramah         4440571            3597076              23.4
                   Aseer         1719950            1259150              36.6
          Eastern Region         3087687            2565462              20.4
               Ar Riyadh         4579570            3858350              18.7
                   Jazan         1187284             873097              36.0
      

#1.4 Disability type mix by region, 2016 (Bindawas & Vennu, Table 3)

In [6]:
# 1.4 Converting Table 3 into the share of each disability type per region, for severe + extreme cases and for all severity levels

t3 = pd.read_excel(f'{RAW}/d2_disability_2016.xlsx', sheet_name='t3_type_by_severity_2016')
print('Raw Table 3:', t3.shape, '| columns:', list(t3.columns))
print('Types:', list(t3['disability_type'].unique()))
print('Severities:', list(t3['severity'].unique()))
print('Cells blank in the source (set to 0):', int(t3['cell_blank_in_source'].sum()))

t3['region'] = t3['region'].map(region_map)
# Values are "per 100,000 persons with disability" in that severity group, so dividing by 1,000 gives a percentage
t3['share_pct'] = t3['per_100k_disabled'] / 1000

# Main version: average of the Severe and Extreme columns
sev_ext = (t3[t3['severity'].isin(['Severe', 'Extreme'])]
           .groupby(['region', 'disability_type'])['share_pct'].mean()
           .rename('share_severe_extreme_pct'))
# Sensitivity version: average of all three severity columns
all3 = (t3.groupby(['region', 'disability_type'])['share_pct'].mean()
          .rename('share_all_levels_pct'))
mix = pd.concat([sev_ext, all3], axis=1).round(2).reset_index()

# Quality checks: 13 regions x 6 types, and each region's shares add up to 100%
sums = mix.groupby('region')[['share_severe_extreme_pct', 'share_all_levels_pct']].sum()
print('\nShare totals per region (should all be 100):')
print(sums.round(2).to_string())
assert len(mix) == 78, 'Expected 13 regions x 6 types'
assert mix['region'].nunique() == 13 and mix['region'].isna().sum() == 0, 'Region mismatch'
assert ((sums - 100).abs() <= 0.1).all().all(), 'A region does not add up to 100%'

mix.to_csv(f'{CLEAN}/typemix_clean.csv', index=False)
print('\nSaved typemix_clean.csv')

view = mix.pivot(index='region', columns='disability_type', values='share_severe_extreme_pct')
print('\nShare of each disability type among severe + extreme cases (%):')
print(view.round(1).to_string())

Raw Table 3: (234, 5) | columns: ['disability_type', 'severity', 'region', 'per_100k_disabled', 'cell_blank_in_source']
Types: ['Mobility', 'Eyesight', 'Hearing', 'Speech & communication', 'Memory', 'Self-care']
Severities: ['Mild', 'Severe', 'Extreme']
Cells blank in the source (set to 0): 27

Share totals per region (should all be 100):
                          share_severe_extreme_pct  share_all_levels_pct
region                                                                  
Al Bahah                                     99.99                100.00
Al Jawf                                     100.00                 99.99
Al Madinah Al Munawwarah                     99.99                 99.99
Al Qaseem                                   100.00                100.01
Ar Riyadh                                   100.00                100.00
Aseer                                       100.01                100.01
Eastern Region                              100.00                100.00
Ha

#1.5 Rehabilitation supply: MOH hospitals offering each rehab service, by region (MOH list, 2021)

In [7]:
# 1.5 Counting MOH hospitals that offer each rehabilitation service in every region (MOH list, 2021)

d3 = pd.read_excel(f'{RAW}/d3_moh_rehab_hospitals.xlsx', sheet_name='hospitals')
print('Raw D3:', d3.shape)

services = ['physical_medicine', 'physiotherapy', 'occupational_therapy',
            'speech_therapy', 'prosthetics_orthotics']
d3['region'] = d3['admin_region'].map(region_map)

# Quality checks on the hospital list before counting
print('Values found in service columns:', sorted(pd.unique(d3[services].values.ravel())))
print('Duplicate hospitals:', d3.duplicated(['region', 'hospital']).sum())
print('Hospitals with no service ticked:', (d3[services].sum(axis=1) == 0).sum())

# One row per region: number of hospitals, and how many offer each service
supply = d3.groupby('region')[services].sum()
supply.insert(0, 'moh_rehab_hospitals', d3.groupby('region').size())
supply = supply.reset_index().sort_values('moh_rehab_hospitals', ascending=False)

# National totals compared with the totals counted from the PDF
totals = supply[['moh_rehab_hospitals'] + services].sum()
expected = pd.Series({'moh_rehab_hospitals': 235, 'physical_medicine': 71, 'physiotherapy': 235,
                      'occupational_therapy': 68, 'speech_therapy': 34, 'prosthetics_orthotics': 17})
print('\nNational totals:')
print(totals.to_string())
assert len(d3) == 235 and d3['region'].isna().sum() == 0, 'Hospital count or region mismatch'
assert supply['region'].nunique() == 13, 'Expected 13 regions'
assert (totals == expected).all(), 'Totals differ from the PDF counts'

supply.to_csv(f'{CLEAN}/supply_clean.csv', index=False)
print('\nSaved supply_clean.csv')
print(supply.to_string(index=False))

Raw D3: (235, 10)
Values found in service columns: [np.int64(0), np.int64(1)]
Duplicate hospitals: 0
Hospitals with no service ticked: 0

National totals:
moh_rehab_hospitals      235
physical_medicine         71
physiotherapy            235
occupational_therapy      68
speech_therapy            34
prosthetics_orthotics     17

Saved supply_clean.csv
                  region  moh_rehab_hospitals  physical_medicine  physiotherapy  occupational_therapy  speech_therapy  prosthetics_orthotics
               Ar Riyadh                   40                  5             40                    15               8                      2
    Makkah Al Mukarramah                   36                 15             36                     9               5                      3
          Eastern Region                   32                 10             32                    10               5                      2
                   Aseer                   24                  8             24    

#1.6 National reference values: 2022 census rates and 2023 survey findings (GASTAT)

In [8]:
# 1.6 Extracting the national reference values from the GASTAT 2023 report (D4), each with its source and PDF page

f = f'{RAW}/d4_gastat_disability_2023.xlsx'
nat    = pd.read_excel(f, sheet_name='national_prevalence')
types  = pd.read_excel(f, sheet_name='types_single_disability')
access = pd.read_excel(f, sheet_name='access_and_cost')

headline = nat[nat['indicator'] == 'Disability + difficulty prevalence']
print('Headline census rates:')
print(headline[['group', 'value_pct', 'source_page']].to_string(index=False))

# Picks exactly one value from a sheet; stops if the label is missing or appears twice
def pick(df, col, label, value_col):
    row = df[df[col] == label]
    assert len(row) == 1, f'Expected one row for {label}, found {len(row)}'
    return float(row[value_col].iloc[0])

adults = types[types['age_group'] == '18+']
national = pd.DataFrame([
    ['rate_total_pct',            pick(headline, 'group', 'Total population', 'value_pct'), 'Census 2022', 'Disability + difficulty, all residents', 13],
    ['rate_saudi_pct',            pick(headline, 'group', 'Saudi', 'value_pct'),            'Census 2022', 'Disability + difficulty, Saudi citizens', 13],
    ['rate_non_saudi_pct',        pick(headline, 'group', 'Non-Saudi', 'value_pct'),        'Census 2022', 'Disability + difficulty, non-Saudi residents', 13],
    ['mobility_share_adults_pct', pick(adults, 'disability_type', 'Mobility (walking or taking stairs)', 'pct_of_single_disability'),
                                  'Survey 2023', 'Mobility share of adults (18+) with one disability', 17],
    ['not_enrolled_rehab_pct',    pick(access, 'item', 'Not enrolled', 'value_pct'),          'Survey 2023', 'Persons with disability (5+) not enrolled in a rehab centre', 23],
    ['reason_distance_pct',       pick(access, 'item', 'Geographical distance', 'value_pct'), 'Survey 2023', 'Top reason for non-enrolment: distance', 24],
    ['reason_no_centres_pct',     pick(access, 'item', 'Lack of centres', 'value_pct'),       'Survey 2023', 'Second reason for non-enrolment: lack of centres', 24],
], columns=['key', 'value', 'source', 'description', 'pdf_page'])

# Quality check: every value matches the figure printed in the PDF
expected = {'rate_total_pct': 4.2, 'rate_saudi_pct': 5.9, 'rate_non_saudi_pct': 1.8,
            'mobility_share_adults_pct': 56.2, 'not_enrolled_rehab_pct': 79.7,
            'reason_distance_pct': 26.7, 'reason_no_centres_pct': 24.4}
for key, value in expected.items():
    assert national.loc[national['key'] == key, 'value'].iloc[0] == value, f'{key} differs from the PDF'

# Context: what the 5.9% means in people, using the 2022 census population from 1.2
saudi_2022 = pop.loc[pop['year'] == 2022, 'saudi'].sum()
print(f'\n5.9% of {saudi_2022:,} Saudis in 2022 = about {saudi_2022 * 0.059:,.0f} people')

national.to_csv(f'{CLEAN}/national_2022.csv', index=False)
print('\nSaved national_2022.csv')
print(national.to_string(index=False))

Headline census rates:
           group  value_pct  source_page
Total population        4.2           13
           Saudi        5.9           13
       Non-Saudi        1.8           13

5.9% of 18,792,262 Saudis in 2022 = about 1,108,743 people

Saved national_2022.csv
                      key  value      source                                                 description  pdf_page
           rate_total_pct    4.2 Census 2022                      Disability + difficulty, all residents        13
           rate_saudi_pct    5.9 Census 2022                     Disability + difficulty, Saudi citizens        13
       rate_non_saudi_pct    1.8 Census 2022                Disability + difficulty, non-Saudi residents        13
mobility_share_adults_pct   56.2 Survey 2023          Mobility share of adults (18+) with one disability        17
   not_enrolled_rehab_pct   79.7 Survey 2023 Persons with disability (5+) not enrolled in a rehab centre        23
      reason_distance_pct   26.7 Surve

#1.7 Health system context by region, 2024 (GASTAT) – background only, not used in calculations

In [9]:
# 1.7 Saving the 2024 regional health system context (GASTAT, D5); five hospital counts stay empty because the source chart hides their labels

f = f'{RAW}/d5_gastat_healthcare_2024.xlsx'
d5    = pd.read_excel(f, sheet_name='regional')
unlab = pd.read_excel(f, sheet_name='hospitals_unlabelled')
print('Raw D5:', d5.shape, '| columns:', list(d5.columns))

context = pd.DataFrame({
    'region':                  d5['region_in_source'].map(region_map),
    'hospitals_all_2024':      d5['hospitals_total_2024'],
    'beds_per_10k_2024':       d5['hospital_beds_per_10k'],
    'phc_per_10k_2024':        d5['phc_centres_per_10k'],
    'private_pharmacies_2024': d5['private_pharmacies'],
})

# Quality checks against the totals published in the infographic
known   = context['hospitals_all_2024'].sum()
missing = context['hospitals_all_2024'].isna().sum()
print(f'Hospitals with a region label: {known:,.0f} | regions without a label: {missing} | unlabelled values: {unlab["hospitals"].tolist()}')
print(f'Known + unlabelled = {known + unlab["hospitals"].sum():,.0f} (published total: 516)')
print(f'Private pharmacies: {context["private_pharmacies_2024"].sum():,} (published total: 10,819)')
print('Regions left empty (label hidden in source):', context.loc[context['hospitals_all_2024'].isna(), 'region'].tolist())
assert context['region'].nunique() == 13 and context['region'].isna().sum() == 0, 'Region mismatch'
assert known + unlab['hospitals'].sum() == 516, 'Hospital total differs from 516'
assert missing == 5, 'Expected exactly 5 unlabelled regions'
assert context['private_pharmacies_2024'].sum() == 10_819, 'Pharmacy total differs from 10,819'
assert context[['beds_per_10k_2024', 'phc_per_10k_2024']].notna().all().all(), 'Beds or PHC value missing'

context.to_csv(f'{CLEAN}/context_clean.csv', index=False)
print('\nSaved context_clean.csv')
print(context.to_string(index=False))

Raw D5: (13, 7) | columns: ['region_std', 'region_in_source', 'hospitals_total_2024', 'hospitals_label_status', 'hospital_beds_per_10k', 'phc_centres_per_10k', 'private_pharmacies']
Hospitals with a region label: 401 | regions without a label: 5 | unlabelled values: [43, 28, 17, 16, 11]
Known + unlabelled = 516 (published total: 516)
Private pharmacies: 10,819 (published total: 10,819)
Regions left empty (label hidden in source): ['Aseer', 'Tabuk', 'Northern Borders', 'Jazan', 'Najran']

Saved context_clean.csv
                  region  hospitals_all_2024  beds_per_10k_2024  phc_per_10k_2024  private_pharmacies_2024
               Ar Riyadh               115.0               23.2              1.65                     3333
    Makkah Al Mukarramah                99.0               20.1              1.37                     2600
Al Madinah Al Munawwarah                33.0               21.2              1.44                      645
               Al Qaseem                25.0           

#1.8 Stage 1 gate: all clean files use the same 13 regions, with no duplicates and no unexpected gaps

In [10]:
# 1.8 Stage 1 gate: reloading every clean file from Drive and confirming they are consistent before analysis begins

official = set(pd.read_csv(f'{CLEAN}/region_lookup.csv')['region'])

# Each file and the columns that should identify a row uniquely
files = {
    'pop_clean.csv':      ['region', 'year'],
    'prev2016_clean.csv': ['region'],
    'typemix_clean.csv':  ['region', 'disability_type'],
    'supply_clean.csv':   ['region'],
    'context_clean.csv':  ['region'],
}

report = []
for name, key in files.items():
    df = pd.read_csv(f'{CLEAN}/{name}')
    regions = set(df['region'])
    report.append({'file': name,
                   'rows': len(df),
                   'regions': len(regions),
                   'same_13_names': regions == official,
                   'duplicate_keys': int(df.duplicated(key).sum()),
                   'empty_cells': int(df.isna().sum().sum())})
report = pd.DataFrame(report)

print('Files in data/clean:', sorted(os.listdir(CLEAN)))
print(report.to_string(index=False))

# The gate: every condition must hold before Stage 2
assert report['same_13_names'].all(), 'A file has different region names'
assert (report['duplicate_keys'] == 0).all(), 'A file has duplicate rows'
expected_empty = {'context_clean.csv': 5}   # the five hidden hospital labels from D5
for _, r in report.iterrows():
    assert r['empty_cells'] == expected_empty.get(r['file'], 0), f"Unexpected empty cells in {r['file']}"
assert len(pd.read_csv(f'{CLEAN}/national_2022.csv')) == 7, 'national_2022.csv should have 7 values'

print('\nGATE 1.8 PASSED – clean data is ready for analysis')

Files in data/clean: ['context_clean.csv', 'national_2022.csv', 'pop_clean.csv', 'prev2016_clean.csv', 'region_lookup.csv', 'supply_clean.csv', 'typemix_clean.csv']
              file  rows  regions  same_13_names  duplicate_keys  empty_cells
     pop_clean.csv   169       13           True               0            0
prev2016_clean.csv    13       13           True               0            0
 typemix_clean.csv    78       13           True               0            0
  supply_clean.csv    13       13           True               0            0
 context_clean.csv    13       13           True               0            5

GATE 1.8 PASSED – clean data is ready for analysis
